# Phishing URL EDA

In [9]:
import pandas as pd
import numpy as np
import plotly.express as px
import tldextract
from urllib.parse import urlsplit, urlunsplit

In [10]:
df = pd.read_csv('../data/malicious_phish.csv')

In [11]:
df.describe()

,url,type
count,651191,651191
unique,641119,4
top,http://style.org.hc360.com/css/detail/mysite/s...,benign
freq,180,428103


This dataset is fairly simple, it consists of 2 columns: The URL itself and the type that it's classified as.
The classification contain:

In [12]:
df.value_counts('type')

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

The Benign URL classification in this context means harmless.

## Volume

The dataset is decently sized with 651191 URL entries, all of which are labeled.
The file itself is 43.5 MB in size.

## Variety
The data in this dataset is structured. The split of URL types looking like this:

In [13]:
p = df['type'].value_counts(normalize=True) * 100

fig = px.bar(
    x=p.index, y=p.values, text=p.values,
    title='Percentage Distribution of URL Types',
    labels={'x': 'URL Type', 'y': 'Percentage (%)'}
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_yaxes(ticksuffix='%')
fig.show()

## Velocity
The dataset is currently only from a single batch. The authors of the dataset mentioned on [kaggle](https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset?resource=download) that they expect to update the dataset annually. 

## Veracity

This dataset seems to be made up of multiple trusted other datasets. The sources can all be found on the [kaggle](https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset?resource=download) page aswell.

In [14]:
ext = tldextract.TLDExtract(suffix_list_urls=())

def split_url(u):
    try:
        p = urlsplit(u if '://' in u else f'//{u}')
        d = ext(p.hostname or '')
        prefix = (f'{p.scheme}://' if p.scheme else '') + (f'{d.subdomain}.' if d.subdomain else '')
        return prefix, d.domain, d.suffix, urlunsplit(('', '', p.path, p.query, p.fragment))
    except (ValueError, UnicodeError, TypeError):
        return ('',) * 4

pp_df = df.copy()
cols = ['prefix', 'website_name', 'suffix', 'sub_pages']
pp_df[cols] = pd.DataFrame(pp_df['url'].map(split_url).tolist(), index=pp_df.index)
pp_df.head()

,url,type,prefix,website_name,suffix,sub_pages
0,br-icloud.com.br,phishing,,br-icloud,com.br,
1,mp3raid.com/music/krizz_kaliko.html,benign,,mp3raid,com,/music/krizz_kaliko.html
2,bopsecrets.org/rexroth/cr/1.htm,benign,,bopsecrets,org,/rexroth/cr/1.htm
3,http://www.garage-pirenne.be/index.php?option=...,defacement,http://www.,garage-pirenne,be,/index.php?option=com_content&view=article&id=...
4,http://adventure-nicaragua.net/index.php?optio...,defacement,http://,adventure-nicaragua,net,/index.php?option=com_mailto&tmpl=component&li...


In [15]:
completeness = (pp_df.replace(r'^\s*$', pd.NA, regex=True).notna().mean() * 100).round(2)
completeness.rename('complete (%)').to_frame()

,complete (%)
url,100.00
type,100.00
prefix,54.15
website_name,99.99
suffix,98.12
sub_pages,96.46


In [16]:
pp_df.to_csv('../data/processed_malicious_phish.csv', index=False)

# Conclusion

This dataset is suitable for training a model for phishing detection. With the processed elements of the URL's extracted the model will be able to find similar features in the non-benign categories of the dataset.
The sources the creator used are all trusted. And the dataset is complete and processable.